# GamaX1 (Aetherion) — Colab GPU Training (Bulk / `--data_dir` path)

**Uses the new `bulk_corpus.py` module**, not a single combined `.txt` file. This is the correct path for a large folder of book files (your 2.2GB+ corpus):

- Never loads the whole corpus into RAM as one giant string — it memory-maps an on-disk int32 token cache.
- Prints progress every 500 books while building the cache, so long runs never look frozen.
- Reuses the cache automatically on re-run (checked by file manifest + tokenizer config), so a Colab disconnect doesn't cost you a re-encode.

**Setup order (important):**
1. `Runtime` → `Change runtime type` → Hardware accelerator = **T4 GPU** (or better) → Save.
2. Run cells top to bottom.
3. Put your book `.txt` files in one folder (subfolders OK, they're found recursively) — do NOT pre-combine them into one file. `bulk_corpus.py` handles that internally, book by book.

## 1. Confirm GPU is attached

In [ ]:
!nvidia-smi

If this errors out or shows no GPU, go back to `Runtime` → `Change runtime type`, pick a GPU, then re-run this cell.

## 2. Mount Google Drive

In [35]:
from google.colab import drive
drive.mount('/content/drive')

import os

PROJECT_ROOT = '/content/drive/MyDrive/Aetherion_GamaX1'

BOOKS_DIR = f'{PROJECT_ROOT}/data/books'
BOOKS2_DIR = f'{PROJECT_ROOT}/data/books2'
CONVERSATION_DIR = f'{PROJECT_ROOT}/data/Conversations-200k'
Discord_Dialogues_DIR = f'{PROJECT_ROOT}/data/Discord-Dialogues'
Reddit_Constructive_DIR = f'{PROJECT_ROOT}/data/Reddit-Constructive'


# Token cache -> Google Drive (FIXED: was previously '/content/...', which is
# Colab's ephemeral local disk. That directory -- including the resumable
# encode_progress.json checkpoint -- is wiped on every runtime disconnect,
# so no matter how good bulk_corpus.py's resume logic is, there was nothing
# left to resume from after any disconnect. Every run silently restarted
# the full encode from file 1. Moving this to Drive fixes that at the root:
# the cache and its checkpoint now survive disconnects like CKPT_DIR always
# has. Named _v2 so this does NOT pick up any old cache dir that may already
# exist on Drive from before this fix (avoids silently trusting a
# possibly-corrupt older cache).
BULK_CACHE_DIR = f'{PROJECT_ROOT}/bulk_cache_multi_v2'

# Checkpoints -> Google Drive (unchanged, already persistent)
CKPT_DIR = f'{PROJECT_ROOT}/checkpoints_bulk_multi'

os.makedirs(BOOKS_DIR, exist_ok=True)
os.makedirs(BOOKS2_DIR, exist_ok=True)
os.makedirs(CONVERSATION_DIR, exist_ok=True)
os.makedirs(Discord_Dialogues_DIR, exist_ok=True)
os.makedirs(Reddit_Constructive_DIR, exist_ok=True)
os.makedirs(BULK_CACHE_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

print('Books dir  :', BOOKS_DIR)
print('Books2 dir :', BOOKS2_DIR)
print('Conversations dir :', CONVERSATION_DIR)
print('Discord dialogues dir :', Discord_Dialogues_DIR)
print('Reddit constructive dir :', Reddit_Constructive_DIR)
print('Bulk cache :', BULK_CACHE_DIR, '(Drive -- persists across disconnects)')
print('Checkpoints:', CKPT_DIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Books dir  : /content/drive/MyDrive/Aetherion_GamaX1/data/books
Books2 dir : /content/drive/MyDrive/Aetherion_GamaX1/data/books2
Conversations dir : /content/drive/MyDrive/Aetherion_GamaX1/data/Conversations-200k
Discord dialogues dir : /content/drive/MyDrive/Aetherion_GamaX1/data/Discord-Dialogues
Reddit constructive dir : /content/drive/MyDrive/Aetherion_GamaX1/data/Reddit-Constructive
Bulk cache : /content/drive/MyDrive/Aetherion_GamaX1/bulk_cache_multi_v2 (Drive -- persists across disconnects)
Checkpoints: /content/drive/MyDrive/Aetherion_GamaX1/checkpoints_bulk_multi


## 3. Get the GamaX1 codebase onto the runtime

**Important:** make sure the version you clone/copy here actually contains `gamax1/bulk_corpus.py` and the `--data_dir` flag in `train.py` — if you pushed those files to GitHub, a plain `git clone`/`git pull` will pick them up. If they only exist locally and were never pushed, Option A (copy from Drive) is safer until you push.

In [36]:
# Option A -- clone without deleting an existing runtime folder
%cd /content
import os
if os.path.exists('gamax1_project'):
    print('Existing gamax1_project found; keeping it. Inspect it before replacing.')
else:
    !git clone https://github.com/mrroy-dev/gamax1.git gamax1_project
%cd /content/gamax1_project
!ls gamax1/


/content
Cloning into 'gamax1_project'...
remote: Enumerating objects: 90, done.
remote: Counting objects: 100% (90/90), done.
remote: Compressing objects: 100% (62/62), done.
remote: Total 90 (delta 48), reused 68 (delta 26), pack-reused 0 (from 0)
Receiving objects: 100% (90/90), 92.43 KiB | 2.72 MiB/s, done.
Resolving deltas: 100% (48/48), done.
/content/gamax1_project
bulk_corpus.py	  generate.py  layers.py  tokenizer.py
compare_dense.py  __init__.py  model.py   train.py


In [ ]:
# Read-only sanity checks: do not modify or delete project files.
from pathlib import Path
assert Path('gamax1/bulk_corpus.py').exists(), 'bulk_corpus.py missing -- wrong code version!'
assert '--data_dir' in Path('gamax1/train.py').read_text(), '--data_dir flag missing from train.py!'
_bulk_src = Path('gamax1/bulk_corpus.py').read_text()
assert 'encode_progress' in _bulk_src and 'PROGRESS_INTERVAL' in _bulk_src, 'Resumable checkpoint logic missing.'
assert '_quarantine_file' in _bulk_src or 'Automatic deletion is disabled' in _bulk_src, 'Safe non-destructive version not detected; review before training.'
# v2 markers: per-source content format (Gutenberg strip / turn-splitting /
# role tags) and the 4-special-token tokenizer. If these are missing, this
# is the OLDER single-eos_id codebase -- still usable, but without the
# per-source fixes (Gutenberg boilerplate, Discord "---" turn-splitting,
# role tags) described in CHANGES_v2.md.
_tok_src = Path('gamax1/tokenizer.py').read_text()
assert 'SOURCE_FORMAT_PROSE' in _bulk_src, 'v2 per-source format logic missing from bulk_corpus.py -- wrong version.'
assert 'SPECIAL_TOKENS' in _tok_src and 'user_id' in _tok_src, 'v2 4-special-token tokenizer missing -- wrong version.'
print('Read-only code checks passed: resumable + safety marker + v2 per-source/special-token markers found.')


## 4. Confirm the books are in place

Upload your `.txt` book files into `BOOKS_DIR` (via the Drive web UI, or `rclone`/`gdown`) beforehand -- as separate files, not pre-merged.

In [32]:
from pathlib import Path

DATA_DIR = Path(BOOKS_DIR).parent  # data/ folder jisme books, wiki, qna sub-folders hain

source_dirs = ["books", "books2", "wiki", "Conversations-200k","Discord-Dialogues","Reddit-Constructive"]
total_files = 0
total_bytes = 0

for name in source_dirs:
    sub_dir = DATA_DIR / name
    if sub_dir.is_dir():
        files = sorted(sub_dir.rglob('*.txt'))
        file_bytes = sum(p.stat().st_size for p in files)
        total_files += len(files)
        total_bytes += file_bytes
        print(f'{name}: {len(files):,} .txt files, {file_bytes / (1024**3):.2f} GB')
    else:
        print(f'{name}: folder not found, skipping')

print(f'\nTotal: {total_files:,} .txt files, {total_bytes / (1024**3):.2f} GB')
assert total_files, f'No .txt files found under {DATA_DIR} (expected books/wiki/qna sub-folders) -- upload your data first.'

books: 4,614 .txt files, 2.07 GB
books2: 12,658 .txt files, 4.60 GB
wiki: 12,161 .txt files, 0.11 GB
Conversations-200k: 416 .txt files, 1.11 GB
Discord-Dialogues: 3,651 .txt files, 0.77 GB
Reddit-Constructive: 803 .txt files, 0.82 GB

Total: 34,303 .txt files, 9.49 GB


## 5. Install dependencies

In [ ]:
!pip install -q torch --extra-index-url https://download.pytorch.org/whl/cu121
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only -- check runtime type!')

CUDA available: True
Device: Tesla T4


## 6. Build/reuse the bulk token cache, then train (auto-resume)

**v2 codebase note:** this version tokenizes each source differently --
`books`/`books2`/`wiki` have their Project Gutenberg license boilerplate
stripped; `Conversations-200k`/`Discord-Dialogues` get real `<|user|>`/
`<|assistant|>` role tags (and a `---`-separated file is split into
separately-bounded exchanges, not glued into one); `Reddit-Constructive`'s
ambiguous `Speaker 0/1:` labels are left untagged but still boundary-split.
See `CHANGES_v2.md` for why.

**Because this changes both the token *content* (not just an added id) and
the tokenizer's `vocab_size` (4 reserved special tokens now, not 1), run
Cell 6a exactly ONCE with `--rebuild_bulk_cache` to force a full re-encode,
into a NEW checkpoint directory. Every session after that, use Cell 6b
(no rebuild, auto-resume) instead.**

First run: this walks every book, encodes it with BPE, and writes the
memory-mapped `tokens.int32.bin` cache -- printing progress every 500
files. Every run after that reuses the cache instantly.


In [ ]:
# Cell 6a -- ONE-TIME: full re-encode with the v2 per-source formats +
# 4-special-token tokenizer, then fresh training. Run this exactly once
# (after confirming Cell 3's sanity check shows the v2 markers), then use
# Cell 6b for every session after.
#
# Do NOT reuse this cell for later resumes -- it forces a full re-encode
# every time via --rebuild_bulk_cache (multi-hour for a multi-GB corpus).

import glob

# NEW checkpoint dir: vocab_size is now bpe_vocab_size + 4 (was +1 in the
# single-eos_id version, which itself was +1 over the original un-tagged
# vocab) -- the embedding/head width changed again, so an old checkpoint
# from any earlier version will NOT load here. Note: --bpe_vocab_size 16000
# below + 4 reserved special tokens = 16,004 total, matching the
# "New vocabulary size: 16004" you already saw and verified once.
CKPT_DIR_V2 = f'{CKPT_DIR}_12L_ctx512_v16k_100m_v2tags'

D_MODEL    = 768
N_HEADS    = 12
N_LAYERS   = 12
N_FEATURES = 3072
BLOCK_SIZE = 512

train_cmd = (
    f'python -m gamax1.train '
    f'--tokenizer bpe '
    f'--data_dir "{DATA_DIR}" '
    f'--bulk_cache_dir "{BULK_CACHE_DIR}" '
    f'--rebuild_bulk_cache '
    f'--d_model {D_MODEL} '
    f'--n_heads {N_HEADS} '
    f'--n_layers {N_LAYERS} '
    f'--n_features {N_FEATURES} '
    f'--block_size {BLOCK_SIZE} '
    f'--batch_size 16 '
    f'--bpe_vocab_size 16000 '
    f'--max_steps 30000 '
    f'--checkpoint_interval 500 '
    f'--out_dir "{CKPT_DIR_V2}"'
).strip()

print('One-time v2 rebuild run -- this will take hours to re-encode the full corpus.')
print(train_cmd)
# !{train_cmd}   # uncomment to actually run


In [ ]:
# Cell 6b -- NORMAL RESUME. Use this for every session after Cell 6a has
# completed at least once. Same CKPT_DIR_V2, no --rebuild_bulk_cache (the
# v2 cache is already on disk and will be reused correctly).

import glob

CKPT_DIR_V2 = f'{CKPT_DIR}_12L_ctx512_v16k_100m_v2tags'

ckpts = sorted(
    glob.glob(f'{CKPT_DIR_V2}/gamax1_step_*.pt'),
    key=lambda p: int(p.rsplit('_', 1)[1].split('.')[0])
)
resume_flag = f'--resume_from "{ckpts[-1]}"' if ckpts else ''
print('Resuming from:', ckpts[-1] if ckpts else '(no checkpoint found -- run Cell 6a first)')

D_MODEL    = 768
N_HEADS    = 12
N_LAYERS   = 12
N_FEATURES = 3072
BLOCK_SIZE = 512

train_cmd = (
    f'python -m gamax1.train '
    f'--tokenizer bpe '
    f'--data_dir "{DATA_DIR}" '
    f'--bulk_cache_dir "{BULK_CACHE_DIR}" '
    f'--d_model {D_MODEL} '
    f'--n_heads {N_HEADS} '
    f'--n_layers {N_LAYERS} '
    f'--n_features {N_FEATURES} '
    f'--block_size {BLOCK_SIZE} '
    f'--batch_size 16 '
    f'--bpe_vocab_size 16000 '
    f'--max_steps 30000 '
    f'--checkpoint_interval 500 '
    f'--out_dir "{CKPT_DIR_V2}" '
    f'{resume_flag}'
).strip()

print(train_cmd)
# !{train_cmd}   # uncomment to actually run


In [ ]:
!{train_cmd}  # runs whichever of Cell 6a / Cell 6b you executed last


In [ ]:
from google.colab import drive

print("Flushing Google Drive and waiting for full cloud sync...")
drive.flush_and_unmount()
print("Drive fully synced and unmounted -- safe to disconnect now.")

Flushing Google Drive and waiting for full cloud sync...
Drive fully synced and unmounted -- safe to disconnect now.


## 7. Generate a sample once training is done (or paused)

v2 adds `--chat` (wraps the prompt with the real `<|user|>`/`<|assistant|>`
ids and stops at the assistant's `eos_id` automatically) alongside the
existing plain-completion mode.


In [ ]:
# Generate from the latest checkpoint in CKPT_DIR_V2.
from pathlib import Path
ckpts = sorted(Path(CKPT_DIR_V2).glob('gamax1_step_*.pt'), key=lambda p: int(p.stem.rsplit('_', 1)[1]))
assert ckpts, f'No step checkpoint found in {CKPT_DIR_V2}; train first.'
LATEST_CKPT = ckpts[-1]
print('Using checkpoint:', LATEST_CKPT)

# Chat-style: wraps the prompt as a real user turn and stops at the
# assistant's eos_id. Only meaningful once trained on the v2 role-tagged
# corpus (i.e. built via Cell 6a's --rebuild_bulk_cache run).
!python -m gamax1.generate --ckpt "{LATEST_CKPT}" --chat \
  --prompt "What is the capital of France?" --max_new_tokens 300 \
  --temperature 0.8 --top_k 40 --repetition_penalty 1.2

# Plain completion (no role wrapping), same as before:
# !python -m gamax1.generate --ckpt "{LATEST_CKPT}" --prompt "who is Prince Andrew" \
#   --max_new_tokens 500 --temperature 0.8 --top_k 40 --repetition_penalty 1.2 --stop_at_eos


In [ ]:
# READ-ONLY cache validation: this cell never deletes or rewrites anything.
from pathlib import Path
import json, os
cache_dir = Path(BULK_CACHE_DIR)
bin_path = cache_dir / 'tokens.int32.bin'
meta_path = cache_dir / 'metadata.json'
progress_path = cache_dir / 'encode_progress.json'
for p in (bin_path, meta_path):
    print(f'{p.name}:', 'PRESENT' if p.exists() else 'MISSING')
if bin_path.exists():
    size = bin_path.stat().st_size
    print('Actual bytes:', f'{size:,}')
    print('Actual tokens:', f'{size // 4:,}')
    print('Byte alignment:', 'OK' if size % 4 == 0 else 'BAD')
if meta_path.exists():
    meta = json.loads(meta_path.read_text())
    print('Metadata keys:', sorted(meta.keys()))
if progress_path.exists():
    print('Progress checkpoint:', 'PRESENT')


Actual bytes: 3821680364
Actual tokens: 955420091


In [ ]:
import os

cache_dir = "/content/drive/MyDrive/Aetherion_GamaX1/bulk_cache_multi_v2"

for filename in os.listdir(cache_dir):
    filepath = os.path.join(cache_dir, filename)
    if os.path.isfile(filepath):
        size = os.path.getsize(filepath)
        print(f"{filename}: {size:,} bytes")

file_index.json: 7,530,518 bytes
tokens.int32.bin: 1,206,219,656 bytes
tokenizer.json: 546,553 bytes
metadata.json: 546,864 bytes


In [ ]:
import os

base_dir = "/content/drive/MyDrive/Aetherion_GamaX1"

for root, dirs, files in os.walk(base_dir):
    for filename in files:
        if filename == "tokens.int32.bin":
            path = os.path.join(root, filename)
            size = os.path.getsize(path)

            print("FOUND CACHE")
            print("Path:", path)
            print("Bytes:", f"{size:,}")
            print("Tokens:", f"{size // 4:,}")
            print("-" * 60)

FOUND CACHE
Path: /content/drive/MyDrive/Aetherion_GamaX1/bulk_cache_multi_v2/tokens.int32.bin
Bytes: 66,759,024
Tokens: 16,689,756
------------------------------------------------------------


In [ ]:
# Compare metadata token count/bytes without changing files.
from pathlib import Path
import json
cache_dir = Path(BULK_CACHE_DIR)
bin_path = cache_dir / 'tokens.int32.bin'
meta_path = cache_dir / 'metadata.json'
if not bin_path.exists():
    print('No token file found; nothing to compare.')
elif not meta_path.exists():
    print('Token file exists but metadata.json is missing -- review before training.')
else:
    actual_bytes = bin_path.stat().st_size
    meta = json.loads(meta_path.read_text())
    expected_bytes = meta.get('token_count', 0) * 4 if 'token_count' in meta else None
    print('Actual bytes:', actual_bytes)
    print('Metadata expected bytes:', expected_bytes)
    if expected_bytes is None:
        print('WARNING: metadata has no token_count; no automatic action taken.')
    elif actual_bytes != expected_bytes:
        print('WARNING: SIZE MISMATCH DETECTED; no files deleted or modified.')
    else:
        print('OK: metadata and token-file byte size agree.')


Tokens: 16689756


## Notes / safety rules

- Progress output is expected every 500 files (`PROGRESS_INTERVAL`).
- Cache and checkpoints live on Google Drive so they can survive runtime disconnects.
- Size mismatches remain visible as warnings; this notebook does not auto-delete files.
- The safe encoder quarantines suspect artifacts with a `.bak` suffix rather than deleting them.
- All validation cells above are read-only. They only inspect files and print warnings.
- Do not run `rm -rf`, `unlink()`, or a rebuild until you have made a backup.
- Before training, ensure the GitHub copy contains the v2 `bulk_corpus.py`/`tokenizer.py`; Cell 3's sanity check verifies this.
- Generation uses the newest `gamax1_step_*.pt` file in `CKPT_DIR_V2`, not an old `gamax1.pt`/`CKPT_DIR_512` path from an earlier codebase version.
- **Run Cell 6a exactly once** (full re-encode + fresh training dir); use Cell 6b for every session after. Running 6a again re-encodes the whole corpus from scratch (multi-hour).
- See `CHANGES_v2.md` (delivered alongside this notebook) for the full list of what changed and why.
